# Pipeline Huấn luyện SCAR (Prompt-free SSPANet + CMSPA M3)
> **Kiến trúc:** Phân đoạn CMR 3 phương thái (**bSSFP/CINE, LGE, T2w**) trên **MyoPS-380** sử dụng **3 ResNetV2 encoders độc lập + SSPANet (RMS Strip Pooling Attention) + CMSPA (Cross-Modal Strip Pathology Attention) + 1 Decoder**.
> Pipeline Prompt-free loại bỏ hoàn toàn phụ thuộc CLIP/MLLM, tối ưu bộ nhớ VRAM và xuất raw logits 4 kênh canonical (`0: background, 1: normal, 2: edema, 3: scar`).

---
### 🔗 Tài nguyên chuẩn bị trên Google Drive:
1. **Pretrained Weights:** `R50-ViT-B_16.npz` (Trọng số ResNetV2 ImageNet21k để khởi tạo cả 3 nhánh encoder).
2. **Bộ dữ liệu MyoPS380:** Thư mục `MyoPS380_dataset` trên Google Drive (`Processed_data` hoặc `Raw_data`).


## Bước 1: Kiểm tra GPU & Kết nối Google Drive
Đảm bảo bạn đã kích hoạt **GPU** trong Colab (`Runtime` -> `Change runtime type` -> Chọn `T4 GPU` hoặc `A100`).


In [ ]:
# 1. Kiểm tra cấu hình GPU
!nvidia-smi

# 2. Kết nối Google Drive để lưu trữ và tải dữ liệu/weights
from google.colab import drive
import os
from pathlib import Path

drive.mount('/content/drive')


## 📥 Bước 2: Clone Repository SCAR mới nhất
Clone mã nguồn SCAR từ GitHub về môi trường Colab.


In [ ]:
%cd /content
!rm -rf SCAR
!git clone https://github.com/thanhquan123hi1/SCAR.git
%cd /content/SCAR
print("Thư mục làm việc hiện tại:", os.getcwd())


## Bước 3: Cài đặt các thư viện cần thiết
Cài đặt trực tiếp từ file `requirements.txt` chuẩn của repository SCAR.


In [ ]:
!pip install -r requirements.txt
import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device Name:", torch.cuda.get_device_name(0))


## Bước 4: Chuẩn bị Pretrained Weights (`R50-ViT-B_16.npz`)
Trọng số này sẽ được nạp đồng bộ vào cả 3 nhánh encoder ResNetV2 (CINE, LGE, T2w) khi huấn luyện.
Hệ thống sẽ ưu tiên copy từ Google Drive cá nhân của bạn, nếu chưa có sẽ tự động tải trực tiếp từ server Google ViT ImageNet21k.


In [ ]:
# Tạo thư mục chứa weights
!mkdir -p /content/SCAR/model/vit_checkpoint/imagenet21k

# CÁCH 1: Copy từ Google Drive cá nhân (Chuẩn đường dẫn Drive của bạn)
drive_vit_path = "/content/drive/MyDrive/NCKH_V2/i_mmseg/ViT_checkpoint/R50-ViT-B_16.npz"

if os.path.exists(drive_vit_path):
    !cp "$drive_vit_path" /content/SCAR/model/vit_checkpoint/imagenet21k/R50-ViT-B_16.npz
    print("✅ Đã copy R50-ViT-B_16.npz từ Google Drive cá nhân!")
else:
    print("⚠️ Không tìm thấy file trong Drive cá nhân. Đang tự động tải từ server Google ViT...")
    # CÁCH 2: Tải trực tiếp từ server Google ViT ImageNet21k
    !wget -c -O /content/SCAR/model/vit_checkpoint/imagenet21k/R50-ViT-B_16.npz https://storage.googleapis.com/vit_models/imagenet21k/R50+ViT-B_16.npz

# Kiểm tra file đã sẵn sàng
!ls -lh /content/SCAR/model/vit_checkpoint/imagenet21k/


## Bước 5: Chuẩn bị Dataset MyoPS380
Sao chép thư mục dataset chuẩn từ Google Drive (`/content/drive/MyDrive/NCKH_V2/i_mmseg/MyoPS380_dataset`) vào môi trường Colab.


In [ ]:
# Tạo thư mục MyoPS380_dataset trong project
!mkdir -p /content/SCAR/MyoPS380_dataset

# Đường dẫn thư mục dataset trên Google Drive cá nhân (Chuẩn đường dẫn của bạn)
drive_dataset_dir = "/content/drive/MyDrive/NCKH_V2/i_mmseg/MyoPS380_dataset"

# Thư mục đích trong project trên Colab
target_dataset_dir = "/content/SCAR/MyoPS380_dataset"

if os.path.exists(drive_dataset_dir):
    print(f"Đang sao chép dataset từ Drive: {drive_dataset_dir}...")
    os.makedirs(target_dataset_dir, exist_ok=True)
    !cp -r "$drive_dataset_dir"/* "$target_dataset_dir"/
    print("✅ Sao chép dataset thành công!")
    print("Các thư mục hiện có:")
    !ls -la "$target_dataset_dir"
else:
    print(f"⚠️ Không tìm thấy thư mục: {drive_dataset_dir}")
    print("👉 Hãy kiểm tra lại xem thư mục MyoPS380_dataset nằm ở thư mục nào trong Drive của bạn.")


## Bước 6: Tiền xử lý dữ liệu (Chỉ chạy nếu bạn dùng `Raw_data`)
- Nếu bạn đã có sẵn `Processed_data`: Hệ thống sẽ tự động nhận diện và bỏ qua bước đóng gói này.
- Nếu bạn dùng `Raw_data`: Script `preprocessing/process_and_save.py` sẽ chuẩn hóa cường độ về `[0, 1]`, bảo toàn test split benchmark (`preprocessing/splits/test_vol.txt`), và chuẩn hóa nhãn về canonical 4-class (`0: background, 1: normal, 2: edema, 3: scar`).


In [ ]:
raw_data_dir = "/content/SCAR/MyoPS380_dataset/Raw_data"
processed_data_dir = "/content/SCAR/MyoPS380_dataset/Processed_data"

if os.path.exists(raw_data_dir) and not os.path.exists(os.path.join(processed_data_dir, "bSSFP", "train_npz")):
    print("Bắt đầu tiền xử lý dữ liệu Raw_data -> Processed_data...")
    !python preprocessing/process_and_save.py \
        --src-path "$raw_data_dir" \
        --dst-path "$processed_data_dir" \
        --test-list preprocessing/splits/test_vol.txt \
        --label-order legacy \
        --normalization unit255 \
        --seed 1234
    print("✅ Tiền xử lý hoàn tất!")
elif os.path.exists(os.path.join(processed_data_dir, "bSSFP", "train_npz")):
    print("✅ Đã có Processed_data sẵn sàng để train!")
else:
    print("⚠️ Không tìm thấy Raw_data hoặc Processed_data. Hãy kiểm tra lại Bước 5!")


## Bước 7: Kiểm tra sơ bộ (Sanity Check)
Chạy thử nghiệm 1 chu trình forward/backward/AdamW đầy đủ trên GPU với AMP trước khi bắt đầu huấn luyện dài hạn.


In [ ]:
!python tools/sanity_check.py --profile testing --device cuda --amp auto --image-size 128


## Bước 8: Huấn luyện Mô hình CMSPA-Net (Ablation M3)
Cấu hình chuẩn tối ưu cho SCAR:
- Kiến trúc: `--ablation M3` (SSPANet + CMSPA)
- Số epoch: `--epochs 300`, Tốc độ học: `--lr 0.001`
- Dữ liệu: `--data-root /content/SCAR/MyoPS380_dataset/Processed_data`
- Splits: `--list-dir data/processed/splits`
- Khởi tạo trọng số: `--pretrained /content/SCAR/model/vit_checkpoint/imagenet21k/R50-ViT-B_16.npz`
- Thư mục lưu kết quả: `--output-dir /content/SCAR/runs/M3`
- Tự động dùng GPU CUDA và Mixed Precision (`--amp auto`)

> 💡 **Lưu ý về Batch Size:** Mặc định `--batch-size 16` chạy tốt trên GPU Colab T4/A100 (15GB–40GB). Nếu dùng GPU nhỏ hơn (như GPU 4GB/6GB), bạn hãy đổi thành `--batch-size 2` kèm `--accum-steps 8`.


In [ ]:
# Xóa thư mục run cũ nếu muốn train lại từ đầu
!rm -rf /content/SCAR/runs/M3

!python training/train.py \
    --config training/config/models/cmspa_net.yaml \
    --ablation M3 \
    --batch-size 16 \
    --accum-steps 1 \
    --lr 0.001 \
    --epochs 300 \
    --output-dir /content/SCAR/runs/M3 \
    --data-root /content/SCAR/MyoPS380_dataset/Processed_data \
    --list-dir data/processed/splits \
    --num-workers 2 \
    --pin-memory \
    --label-order canonical \
    --pretrained /content/SCAR/model/vit_checkpoint/imagenet21k/R50-ViT-B_16.npz


## Bước 9: Sao lưu Checkpoints & Kết quả vào Google Drive
Sao lưu toàn bộ kết quả huấn luyện (gồm checkpoints `best.pth`, `last.pth`, logs và file `metrics.csv`) sang Google Drive cá nhân.


In [ ]:
backup_drive_dir = "/content/drive/MyDrive/NCKH_V2/SCAR_runs/M3"
os.makedirs(backup_drive_dir, exist_ok=True)

!cp -r /content/SCAR/runs/M3/* "$backup_drive_dir/"
print(f"✅ Đã sao lưu toàn bộ kết quả runs vào: {backup_drive_dir}")
!ls -lh "$backup_drive_dir"


## Bước 10: Đánh giá mô hình trên tập Test Volume (3D Testing)
Thực hiện dự đoán và tính toán đầy đủ các chỉ số định lượng (Dice, HD95, ASD) trên toàn bộ các ca bệnh kiểm thử 3D trong `test_vol.txt`, đồng thời xuất các file dự đoán NIfTI để trực quan hóa giải phẫu.


In [ ]:
!python training/evaluate.py \
    --checkpoint /content/SCAR/runs/M3/best.pth \
    --data-root /content/SCAR/MyoPS380_dataset/Processed_data \
    --split test_vol \
    --device cuda \
    --amp auto \
    --batch-size 8 \
    --label-order canonical \
    --save-predictions


## Bước 11: Trực quan hóa Biểu đồ Huấn luyện
Vẽ biểu đồ Loss (Train / Val) và chỉ số hiệu năng phân đoạn (Dice Score, IoU) qua từng epoch từ `metrics.csv`.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

metrics_path = Path("/content/SCAR/runs/M3/metrics.csv")
if not metrics_path.is_file():
    metrics_path = Path("/content/drive/MyDrive/NCKH_V2/SCAR_runs/M3/metrics.csv")

if metrics_path.is_file():
    metrics = pd.read_csv(metrics_path)
    display(metrics.tail())

    plt.figure(figsize=(14, 5))
    
    # Đồ thị Loss
    plt.subplot(1, 2, 1)
    plt.plot(metrics["epoch"], metrics["train/loss"], label="Train Loss", color="blue")
    if "val/loss" in metrics.columns:
        plt.plot(metrics["epoch"], metrics["val/loss"], label="Val Loss", color="orange")
    plt.title("Đường cong mất mát (Loss Curve)")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True, linestyle="--", alpha=0.6)

    # Đồ thị Dice & IoU
    plt.subplot(1, 2, 2)
    if "val/mean_dice" in metrics.columns:
        plt.plot(metrics["epoch"], metrics["val/mean_dice"], label="Val Mean Dice", color="green")
    if "val/mean_iou" in metrics.columns:
        plt.plot(metrics["epoch"], metrics["val/mean_iou"], label="Val Mean IoU", color="purple")
    plt.title("Hiệu năng phân đoạn (Validation Metrics)")
    plt.xlabel("Epoch")
    plt.ylabel("Score")
    plt.legend()
    plt.grid(True, linestyle="--", alpha=0.6)

    plt.tight_layout()
    plt.show()
else:
    print("⚠️ Chưa tìm thấy file metrics.csv tại:", metrics_path)
